# Simple Subsystem Demonstration

This notebook demonstrates a single quantum subsystem with a custom gate pattern.

In [ ]:
# Import necessary modules
import sys
sys.path.append('..')

from modul.circuit import Circuit
import numpy as np
from qiskit import QuantumCircuit

## Custom Subsystem with PP-H-PP-H-PP Pattern

We create a subsystem with 3 qubits using the gate pattern: **P-P-H-P-P-H-P-P**

In [ ]:
# Create a circuit with 3 qubits
circuit = Circuit(total_qubits=3)
qc = circuit.get_circuit()

# Generate random phase values
phases = np.random.rand(3, 6) * 2 * np.pi

# Apply PP-H-PP-H-PP pattern to all 3 qubits
for qubit in range(3):
    # PP
    qc.p(phases[qubit, 0], qubit)
    qc.p(phases[qubit, 1], qubit)
    # H
    qc.h(qubit)
    # PP
    qc.p(phases[qubit, 2], qubit)
    qc.p(phases[qubit, 3], qubit)
    # H
    qc.h(qubit)
    # PP
    qc.p(phases[qubit, 4], qubit)
    qc.p(phases[qubit, 5], qubit)

# Display the circuit
print("Subsystem with 3 qubits:")
print("Gate pattern: P-P-H-P-P-H-P-P")
print("\nCircuit visualization:")
print(qc.draw(output='text'))

## Input and Training Matrices

Now we'll create two matrices that represent different aspects of our quantum system:

In [ ]:
# Input Matrix - Identity Matrix 3x3
# This represents the initial state or input configuration
input_matrix = np.eye(3)
print("Input Matrix (3x3):")
print("This identity matrix represents the initial input state")
print(input_matrix)

print("\n" + "-"*50 + "\n")

# Training Matrix - Random Matrix 3x3
# This represents the training parameters or weights
training_matrix = np.random.rand(3, 3)
print("Training Matrix (3x3):")
print("This random matrix represents the training parameters")
print(training_matrix)

## Circuit with Input and Training Matrices

Now we'll create a new circuit where:
- **IP** (Input Matrix values) are used for the first phase gate in each pair
- **TP** (Training Matrix values) are used for the second phase gate in each pair

Pattern: **IP-TP-H-IP-TP-H-IP-TP**

In [ ]:
# Create a new circuit with matrix values
matrix_circuit = Circuit(total_qubits=3)
qc_matrix = matrix_circuit.get_circuit()

# Reshape matrices to get values for each qubit
# Each qubit gets one row from the matrices
for qubit in range(3):
    # IP1-TP1 (first pair from first column)
    qc_matrix.p(input_matrix[qubit, 0], qubit)
    qc_matrix.p(training_matrix[qubit, 0], qubit)
    
    # H (first Hadamard)
    qc_matrix.h(qubit)
    
    # IP2-TP2 (second pair from second column)
    qc_matrix.p(input_matrix[qubit, 1], qubit)
    qc_matrix.p(training_matrix[qubit, 1], qubit)
    
    # H (second Hadamard)
    qc_matrix.h(qubit)
    
    # IP3-TP3 (third pair from third column)
    qc_matrix.p(input_matrix[qubit, 2], qubit)
    qc_matrix.p(training_matrix[qubit, 2], qubit)

print("Circuit with matrix values:")
print("Pattern: IP-TP-H-IP-TP-H-IP-TP")
print(f"\\nQubit 0: IP={input_matrix[0]}, TP={training_matrix[0]}\")")
print(f"Qubit 1: IP={input_matrix[1]}, TP={training_matrix[1]}\")")
print(f"Qubit 2: IP={input_matrix[2]}, TP={training_matrix[2]}\")")

print("\\nCircuit visualization:")
print(qc_matrix.draw(output='text'))

## Measurement and Probability Distribution

Now we'll measure all qubits and show the probability distribution of the quantum states.

In [ ]:
# Add measurements to all qubits
qc_matrix.measure_all()

# Import necessary modules for simulation
from qiskit import transpile
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt

# Create a simulator
simulator = AerSimulator()

# Transpile the circuit for the simulator
compiled_circuit = transpile(qc_matrix, simulator)

# Run the simulation with 1000 shots
job = simulator.run(compiled_circuit, shots=1000)
result = job.result()
counts = result.get_counts(compiled_circuit)

# Display the measurement results
print("Measurement Results (1000 shots):")
print("-" * 40)
for state, count in sorted(counts.items()):
    probability = count / 1000
    print(f"State |{state}>: {count} counts ({probability:.2%})")

# Create a bar plot of the probability distribution
plt.figure(figsize=(10, 6))
states = list(counts.keys())
probabilities = [count/1000 for count in counts.values()]

plt.bar(states, probabilities)
plt.xlabel('Quantum States')
plt.ylabel('Probability')
plt.title('Probability Distribution of Quantum States')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Add percentage labels on bars
for i, (state, prob) in enumerate(zip(states, probabilities)):
    plt.text(i, prob + 0.01, f'{prob:.1%}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\\nTotal number of unique states measured: {len(counts)}")
print(f"Most probable state: |{max(counts, key=counts.get)}> with probability {max(counts.values())/1000:.2%}")

## Target State Definition

We'll use the most probable state from our measurement as the target state for optimization.

In [ ]:
# Define the target state as the most probable state from initial measurement
target_state = max(counts, key=counts.get)
target_prob_initial = counts[target_state] / 1000

print(f"Target State: |{target_state}>")
print(f"Initial probability: {target_prob_initial:.2%}")
print(f"\nGoal: Optimize the training matrix to maximize the probability of |{target_state}>")

## Training with TensorFlow

Now we'll use TensorFlow to optimize the training matrix phases to maximize the probability of our target state.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from qiskit.quantum_info import Statevector

# Convert target state string to index
target_index = int(target_state.replace(' ', ''), 2)

# Store optimization history
loss_history = []
prob_history = []

def create_circuit_with_phases(input_mat, training_phases_flat):
    """Create circuit with given phase matrices"""
    # Reshape flat array back to matrix
    training_phases_mat = training_phases_flat.reshape(3, 3)
    
    qc = QuantumCircuit(3)
    
    for qubit in range(3):
        # IP1-TP1
        qc.p(float(input_mat[qubit, 0]), qubit)
        qc.p(float(training_phases_mat[qubit, 0]), qubit)
        # H
        qc.h(qubit)
        # IP2-TP2
        qc.p(float(input_mat[qubit, 1]), qubit)
        qc.p(float(training_phases_mat[qubit, 1]), qubit)
        # H
        qc.h(qubit)
        # IP3-TP3
        qc.p(float(input_mat[qubit, 2]), qubit)
        qc.p(float(training_phases_mat[qubit, 2]), qubit)
    
    return qc

def objective_function(training_phases_flat):
    """Objective function to minimize (negative probability of target state)"""
    # Create circuit with current phases
    qc_train = create_circuit_with_phases(input_matrix, training_phases_flat)
    
    # Get statevector
    state = Statevector.from_instruction(qc_train)
    probs = state.probabilities()
    
    # Get probability of target state
    target_prob = probs[target_index]
    
    # Store for history
    loss = -np.log(target_prob + 1e-10)
    loss_history.append(loss)
    prob_history.append(target_prob)
    
    # Return negative probability (we want to minimize this)
    return -target_prob

# Initial phases (flatten the training matrix)
initial_phases = training_matrix.flatten() * 2 * np.pi

print("Optimization Progress:")
print("-" * 50)

# Callback to print progress
iteration_count = 0
def callback(xk):
    global iteration_count
    if iteration_count % 20 == 0:
        current_prob = prob_history[-1] if prob_history else 0
        current_loss = loss_history[-1] if loss_history else 0
        print(f"Iteration {iteration_count}: Target state probability = {current_prob:.4f}, Loss = {current_loss:.4f}")
    iteration_count += 1

# Optimize using scipy
result = minimize(
    objective_function,
    initial_phases,
    method='L-BFGS-B',
    bounds=[(0, 2*np.pi) for _ in range(9)],  # 9 phases (3x3 matrix)
    callback=callback,
    options={'maxiter': 100}
)

# Get final results
optimized_phases = result.x
final_training_matrix = optimized_phases.reshape(3, 3) / (2 * np.pi)

# Create final circuit
qc_final = create_circuit_with_phases(input_matrix, optimized_phases)
final_state = Statevector.from_instruction(qc_final)
final_probs = final_state.probabilities()
final_target_prob = float(final_probs[target_index])

print(f"\nOptimization Complete!")
print(f"Initial probability: {target_prob_initial:.2%}")
print(f"Final probability: {final_target_prob:.2%}")
print(f"Improvement: {(final_target_prob - target_prob_initial)*100:.2f} percentage points")
print(f"Optimization success: {result.success}")
print(f"Number of iterations: {result.nit}")

## Results: New Probability Distribution, Optimization Function, and New Matrix

In [ ]:
# 1. New Probability Distribution
print("1. NEW PROBABILITY DISTRIBUTION")
print("=" * 50)

# Measure the optimized circuit
qc_final.measure_all()
compiled_final = transpile(qc_final, simulator)
job_final = simulator.run(compiled_final, shots=1000)
result_final = job_final.result()
counts_final = result_final.get_counts(compiled_final)

# Display comparison
print("\\nState Probabilities (Before → After):")
print("-" * 40)
all_states = sorted(set(list(counts.keys()) + list(counts_final.keys())))
for state in all_states:
    prob_before = counts.get(state, 0) / 1000
    prob_after = counts_final.get(state, 0) / 1000
    change = prob_after - prob_before
    if state == target_state:
        print(f"**|{state}>: {prob_before:.2%} → {prob_after:.2%} (Δ = {change:+.2%}) [TARGET]**")
    else:
        print(f"  |{state}>: {prob_before:.2%} → {prob_after:.2%} (Δ = {change:+.2%})")

# 2. Optimization Function (Loss Curve)
print("\\n\\n2. OPTIMIZATION FUNCTION (Loss Curve)")
print("=" * 50)

plt.figure(figsize=(10, 5))
plt.plot(loss_history, 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss (-log probability)')
plt.title('Training Loss Over Time')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Initial loss: {loss_history[0]:.4f}")
print(f"Final loss: {loss_history[-1]:.4f}")
print(f"Loss reduction: {loss_history[0] - loss_history[-1]:.4f}")

# 3. New Training Matrix
print("\\n\\n3. NEW TRAINING MATRIX")
print("=" * 50)

print("\\nOriginal Training Matrix:")
print(training_matrix)

print("\\nOptimized Training Matrix (normalized to [0, 1]):")
print(final_training_matrix)

print("\\nChange in Training Matrix:")
change_matrix = final_training_matrix - training_matrix
print(change_matrix)

# Visualize the optimized probability distribution
plt.figure(figsize=(12, 6))

# Before optimization
plt.subplot(1, 2, 1)
states_before = list(counts.keys())
probs_before = [count/1000 for count in counts.values()]
bars1 = plt.bar(states_before, probs_before, color='lightblue', edgecolor='black')
# Highlight target state
target_idx = states_before.index(target_state) if target_state in states_before else -1
if target_idx >= 0:
    bars1[target_idx].set_color('orange')
plt.xlabel('Quantum States')
plt.ylabel('Probability')
plt.title('Before Optimization')
plt.xticks(rotation=45)
plt.ylim(0, max(max(probs_before), max([count/1000 for count in counts_final.values()])) * 1.2)

# After optimization
plt.subplot(1, 2, 2)
states_after = list(counts_final.keys())
probs_after = [count/1000 for count in counts_final.values()]
bars2 = plt.bar(states_after, probs_after, color='lightgreen', edgecolor='black')
# Highlight target state
target_idx = states_after.index(target_state) if target_state in states_after else -1
if target_idx >= 0:
    bars2[target_idx].set_color('darkgreen')
plt.xlabel('Quantum States')
plt.ylabel('Probability')
plt.title('After Optimization')
plt.xticks(rotation=45)
plt.ylim(0, max(max(probs_before), max(probs_after)) * 1.2)

plt.tight_layout()
plt.show()

## Final Optimized Circuit and Probability Distribution

In [ ]:
# Create and display the final optimized circuit
print("FINAL OPTIMIZED CIRCUIT")
print("=" * 60)
print("Pattern: IP-TP(optimized)-H-IP-TP(optimized)-H-IP-TP(optimized)")
print("\nCircuit visualization:")
print(qc_final.draw(output='text'))

# Run a fresh simulation with the optimized circuit
qc_final_fresh = create_circuit_with_phases(input_matrix, optimized_phases)
qc_final_fresh.measure_all()

# Simulate
compiled_final_fresh = transpile(qc_final_fresh, simulator)
job_final_fresh = simulator.run(compiled_final_fresh, shots=1000)
result_final_fresh = job_final_fresh.result()
counts_final_fresh = result_final_fresh.get_counts(compiled_final_fresh)

print("\n\nFINAL PROBABILITY DISTRIBUTION")
print("=" * 60)
print("\nMeasurement Results (1000 shots):")
print("-" * 40)

# Sort states by probability
sorted_states = sorted(counts_final_fresh.items(), key=lambda x: x[1], reverse=True)

for state, count in sorted_states:
    probability = count / 1000
    if state == target_state:
        print(f"**|{state}>: {count} counts ({probability:.2%}) [TARGET STATE]**")
    else:
        print(f"  |{state}>: {count} counts ({probability:.2%})")

# Create visualization
plt.figure(figsize=(10, 6))
states = [item[0] for item in sorted_states]
probabilities = [item[1]/1000 for item in sorted_states]

bars = plt.bar(states, probabilities, color='skyblue', edgecolor='black')

# Highlight target state
if target_state in states:
    target_idx = states.index(target_state)
    bars[target_idx].set_color('darkgreen')
    bars[target_idx].set_edgecolor('black')
    bars[target_idx].set_linewidth(2)

plt.xlabel('Quantum States', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title('Final Probability Distribution After Optimization', fontsize=14)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Add percentage labels on bars
for i, (state, prob) in enumerate(zip(states, probabilities)):
    if state == target_state:
        plt.text(i, prob + 0.01, f'{prob:.1%}', ha='center', va='bottom', 
                fontweight='bold', fontsize=11)
    else:
        plt.text(i, prob + 0.01, f'{prob:.1%}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\nSummary:")
print(f"- Target state |{target_state}> probability: {counts_final_fresh.get(target_state, 0)/1000:.1%}")
print(f"- Number of unique states measured: {len(counts_final_fresh)}")
print(f"- Most probable state: |{max(counts_final_fresh, key=counts_final_fresh.get)}>")